In [ ]:
import os

print(os.getcwd())

os.chdir('path/to/your/directory')

In [ ]:
conda info --envs

In [ ]:
#import libs
import os
import scvelo as scv
import pandas as pd
import scanpy as sc
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.graphics.mosaicplot as mplt
import rpy2.robjects as robjects
import warnings
import cellrank as cr
from tqdm import tqdm
import loompy as lmp
from scipy import stats


scv.settings.verbosity = 3
scv.settings.set_figure_params('scvelo', facecolor='white', dpi=100, frameon=False)
cr.settings.verbosity = 2

In [ ]:
import warnings
warnings.simplefilter("ignore", category=DeprecationWarning)

In [ ]:
adata = sc.read_h5ad('recluster.h5ad')

In [ ]:
adata1 = sc.read_h5ad('vehicle.h5ad')
adata2 = sc.read_h5ad('pth.h5ad')

In [ ]:
adata1

In [ ]:
# load loom files for spliced/unspliced matrices for each sample:
ldata1 = sc.read('Matrix/HKd29/velocyto/possorted_genome_bam_SPKT1.loom', cache=True)
ldata2 = sc.read('Matrix/HKd30/velocyto/possorted_genome_bam_47EKK.loom', cache=True)
ldata3 = sc.read('Matrix/HKd31/velocyto/possorted_genome_bam_WQOD7.loom', cache=True)
ldata4 = sc.read('Matrix/HKd32/velocyto/possorted_genome_bam_9ICS9.loom', cache=True)



In [ ]:
#rename the barcodes to match the matrix from Seurat
barcodes = [bc.split(':')[1] for bc in ldata1.obs.index.tolist()]
barcodes = ['veh2d_' + bc[0:len(bc)-1] for bc in barcodes]
ldata1.obs.index = barcodes

In [ ]:
barcodes = [bc.split(':')[1] for bc in ldata2.obs.index.tolist()]
barcodes = ['pth2d_' + bc[0:len(bc)-1] for bc in barcodes]
ldata2.obs.index = barcodes

In [ ]:
barcodes = [bc.split(':')[1] for bc in ldata3.obs.index.tolist()]
barcodes = ['veh10d_' + bc[0:len(bc)-1] for bc in barcodes]
ldata3.obs.index = barcodes

In [ ]:
barcodes = [bc.split(':')[1] for bc in ldata4.obs.index.tolist()]
barcodes = ['pth10d_' + bc[0:len(bc)-1] for bc in barcodes]
ldata4.obs.index = barcodes

In [ ]:
#check barcode names
ldata3.obs.index

In [ ]:
#cheeck barcode name from seurat matrix
adata2.obs.index

In [ ]:
# make variable names unique
ldata1.var_names_make_unique()
ldata2.var_names_make_unique()
ldata3.var_names_make_unique()
ldata4.var_names_make_unique()


In [ ]:
### saving barcode into the metadata
adata.obs['cell_barcode'] = adata.obs.index

In [ ]:
# concatenate the three loom
ldata = ldata1.concatenate([ldata2, ldata3, ldata4])

In [ ]:
ldata1 = ldata1.concatenate([ldata3])

In [ ]:
ldata2 = ldata2.concatenate([ldata4])

In [ ]:
# merge matrices into the original adata object
adata = scv.utils.merge(adata, ldata)

In [ ]:
adata1 = scv.utils.merge(adata1, ldata1)

In [ ]:
adata2 = scv.utils.merge(adata2, ldata2)

In [ ]:
# plot umap to check
sc.pl.umap(adata1, color=['cluster_ids', 'orig.ident'], frameon=False, legend_loc='on data', title='')

In [ ]:
adata1.obs['cluster_ids'].cat.categories

In [ ]:

# Define the desired order
desired_order = ["CAR1", "CAR2", "CAR3", "CAR4","CAR5", "CAR6","Pre-osteoblast", "Mature osteoblast", "Periosteal1", "Periosteal2", "Periosteal3", "Endothelial", "Pericyte1", "Pericyte2", "Myogenic"]

# Convert to categorical with specified order
adata1.obs['cluster_ids'] = pd.Categorical(
    adata1.obs['cluster_ids'],  # Replace with your actual cluster column (e.g., 'leiden')
    categories=desired_order,
    ordered=True
)


In [ ]:

# Define the desired order
desired_order = ["CAR1", "CAR2", "CAR3", "CAR4","CAR5", "CAR6", "Pre-osteoblast", "Mature osteoblast", "Periosteal1", "Periosteal2", "Periosteal3", "Endothelial", "Pericyte1", "Pericyte2", "Myogenic"]

# Convert to categorical with specified order
adata2.obs['recluster_ids'] = pd.Categorical(
    adata2.obs['recluster_ids'],  # Replace with your actual cluster column (e.g., 'leiden')
    categories=desired_order,
    ordered=True
)


In [ ]:
scv.pl.proportions(adata1, groupby='cluster_ids')

In [ ]:
# subsetting our clusters of interest
cur_celltypes = ['OsteoCAR1', 'OsteoCAR2', 'OsteoCAR3', 'OsteoCAR4', 'AdipoCAR', 'Pre-osteoblast', 'Mature osteoblast']

adata_subset = adata[adata.obs['cluster_ids'].isin(cur_celltypes)]

In [ ]:
# subsetting our clusters of interest
cur_celltypes = ['CAR1', 'CAR2', 'CAR3', 'CAR4', 'CAR5', 'CAR6','Pre-osteoblast']

adata1_subset = adata1[adata1.obs['cluster_ids'].isin(cur_celltypes)]

In [ ]:
# subsetting our clusters of interest
cur_celltypes = ['CAR1', 'CAR2', 'CAR3', 'CAR4', 'CAR5', 'CAR6','Pre-osteoblast']

adata2_subset = adata2[adata2.obs['cluster_ids'].isin(cur_celltypes)]

In [ ]:
# plotting the new subsetted clusters
sc.pl.umap(adata1_subset, color=['cluster_ids', 'orig.ident'], frameon=False, title=['', ''])

In [ ]:
scv.pl.proportions(adata2_subset, groupby='cluster_ids', save='pthproportions_full.pdf')

In [ ]:
# pre-process
sc.pp.neighbors(adata, n_neighbors=30, use_rep='X_pca')
scv.pp.filter_and_normalize(adata)

In [ ]:
# pre-process
sc.pp.neighbors(adata1_subset, n_neighbors=30, use_rep='X_pca')
scv.pp.filter_and_normalize(adata1_subset)

In [ ]:
# pre-process
sc.pp.neighbors(adata2_subset, n_neighbors=30, use_rep='X_pca')
scv.pp.filter_and_normalize(adata2_subset)

In [ ]:
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

In [ ]:
scv.pp.moments(adata1_subset, n_pcs=30, n_neighbors=30)

In [ ]:
scv.pp.moments(adata2_subset, n_pcs=30, n_neighbors=30)

In [ ]:
#### dynamical modeling
scv.tl.recover_dynamics(adata1_subset, n_jobs=12)

In [ ]:
#### dynamical modeling
scv.tl.recover_dynamics(adata2_subset, n_jobs=12)

In [ ]:

scv.tl.velocity(adata1_subset, mode = 'dynamical')
scv.tl.velocity_graph(adata1_subset)
scv.pl.velocity_embedding_stream(adata1_subset, basis = 'umap', color = 'cluster_ids')

In [ ]:

scv.pl.velocity_embedding_stream(adata1_subset, basis = 'umap', color = 'cluster_ids', save='dynamicalveh_full3.png')

In [ ]:

scv.tl.velocity(adata2_subset, mode = 'dynamical')
scv.tl.velocity_graph(adata2_subset)
scv.pl.velocity_embedding_stream(adata2_subset, basis = 'umap', color = 'cluster_ids')

In [ ]:

scv.pl.velocity_embedding_stream(adata2_subset, basis = 'umap', color = 'cluster_ids', save='dynamicalpth_full3.png')

In [ ]:
scv.tl.velocity_pseudotime(adata1_subset)
scv.pl.scatter(adata1_subset, color='velocity_pseudotime', cmap='gnuplot', size=1)

In [ ]:
scv.tl.velocity_pseudotime(adata2_subset)
scv.pl.scatter(adata2_subset, color='velocity_pseudotime', cmap='gnuplot', size=1)

In [ ]:
scv.tl.latent_time(adata1_subset)
scv.pl.scatter(adata1_subset, color='latent_time', color_map='gnuplot', size=20)

In [ ]:
scv.tl.latent_time(adata2_subset)
scv.pl.scatter(adata2_subset, color='latent_time', color_map='gnuplot', size=20)

In [ ]:
scv.pl.scatter(adata1_subset, var_names=['Malat1', 'Rora'], color='cluster_ids')

In [ ]:
scv.pl.scatter(adata2_subset, var_names=['Malat1', 'Rora'], color='cluster_ids')

In [ ]:
# plot velocity of a selected gene
scv.pl.velocity(adata, var_names=['Pecam1'], color='cluster_ids')

In [ ]:
# plot velocity of a selected gene
scv.pl.velocity(adata1_subset, var_names=['Malat1', 'Rora'], color='cluster_ids')

In [ ]:
# plot velocity of a selected gene
scv.pl.velocity(adata2_subset, var_names=['Mata1', 'Rora'], color='cluster_ids')

In [ ]:
scv.tl.rank_velocity_genes(adata1_subset, groupby='cluster_ids', min_corr=0.3)
df1 = scv.get_df(adata1_subset, 'rank_velocity_genes/names')
df1.head(5)

In [ ]:
for cluster in ['OsteoCAR1', 'OsteoCAR2', 'OsteoCAR3', 'OsteoCAR4', 'AdipoCAR', 'Pre-osteoblast', 'Mature osteoblast']:
    scv.pl.scatter(adata1_subset, df1[cluster][:3], ylabel=cluster, frameon=False, color = 'cluster_ids', save=f'{cluster}.png')

In [ ]:
scv.tl.rank_velocity_genes(adata2_subset, groupby='cluster_ids', min_corr=0.3)
df2 = scv.get_df(adata2_subset, 'rank_velocity_genes/names')
df2.head(5)

In [ ]:
for cluster in ['OsteoCAR1', 'OsteoCAR2', 'OsteoCAR3', 'OsteoCAR4', 'AdipoCAR', 'Pre-osteoblast', 'Mature osteoblast']:
    scv.pl.scatter(adata2_subset, df2[cluster][:3], ylabel=cluster, frameon=False, color = 'cluster_ids')

In [ ]:
scv.tl.rank_dynamical_genes(adata1_subset, groupby='cluster_ids')
df1 = scv.get_df(adata1_subset, 'rank_dynamical_genes/names')
df1.head(5)

In [ ]:
for cluster in ['OsteoCAR1', 'OsteoCAR2', 'OsteoCAR3', 'OsteoCAR4', 'AdipoCAR', 'Pre-osteoblast', 'Mature osteoblast']:
    scv.pl.scatter(adata1_subset, df1[cluster][:4], ylabel=cluster, frameon=False, color = 'cluster_ids', save=f'veh{cluster}.png')

In [ ]:
scv.tl.rank_dynamical_genes(adata2_subset, groupby='cluster_ids')
df2 = scv.get_df(adata2_subset, 'rank_dynamical_genes/names')
df2.head(5)

In [ ]:
for cluster in ['OsteoCAR1', 'OsteoCAR2', 'OsteoCAR3', 'OsteoCAR4', 'AdipoCAR', 'Pre-osteoblast', 'Mature osteoblast']:
    scv.pl.scatter(adata2_subset, df2[cluster][:4], ylabel=cluster, frameon=False, color = 'cluster_ids', save=f'pth{cluster}.png')

In [ ]:
scv.tl.velocity_confidence(adata)
keys = 'velocity_length', 'velocity_confidence'
scv.pl.scatter(adata, c = keys, perc = [5,95])

In [ ]:
scv.tl.velocity_pseudotime(adata)
scv.pl.scatter(adata, color = 'velocity_pseudotime')

In [ ]:
scv.pl.scatter(adata1_subset, 'Cxcl12', color=['cluster_ids', 'velocity'], add_outline='AdipoCAR')

In [ ]:

scv.tl.velocity_pseudotime(adata_subset, root_key='root_key')
scv.pl.scatter(adata_subset, color=["root_cells", "end_points"])

In [ ]:
scv.tl.recover_dynamics(adata_subset, n_jobs = 20)
scv.tl.velocity(adata_subset, mode = 'dynamical', root_key='root_key')
scv.tl.velocity_graph(adata_subset)
scv.pl.velocity_embedding_stream(adata_subset, basis = 'umap', color = 'new_clusters', arrow_size=2)

In [ ]:
scv.tl.latent_time(adata1_subset)
scv.pl.scatter(adata1_subset, color='latent_time', color_map='gnuplot', size=80)

In [ ]:
scv.tl.latent_time(adata2_subset)
scv.pl.scatter(adata2_subset, color='latent_time', color_map='gnuplot', size=80)

In [ ]:
sc.tl.leiden(adata_subset)

In [ ]:
#assigning your root cluster

df1 = pd.DataFrame(index=adata1_subset.obs_names).reset_index()
adata1_subset.uns['root_key'] = df1.index[df1['index'].isin(adata1_subset.obs_names[adata1_subset.obs['cluster_ids'] == 'Mature osteoblast'])][0]

scv.tl.latent_time(adata1_subset, root_key='root_key')

In [ ]:
#assigning your root cluster

df2 = pd.DataFrame(index=adata2_subset.obs_names).reset_index()
adata2_subset.uns['root_key'] = df2.index[df2['index'].isin(adata2_subset.obs_names[adata2_subset.obs['cluster_ids'] == 'Mature osteoblast'])][0]

scv.tl.latent_time(adata2_subset, root_key='root_key')

In [ ]:
# Define terminal cell cluster manually
adata1_subset.obs["terminal_cells"] = adata1_subset.obs["cluster_ids"].isin(["AdipoCAR"])

# Assign terminal cells for latent time estimation
terminal_indices = adata1_subset.obs["terminal_cells"].values

# Compute latent time with manually set terminal cells
scv.tl.latent_time(adata1_subset, end_key="terminal_cells")


In [ ]:
# Define terminal cell cluster manually
adata2_subset.obs["terminal_cells"] = adata2_subset.obs["cluster_ids"].isin(["AdipoCAR"])

# Assign terminal cells for latent time estimation
terminal_indices = adata2_subset.obs["terminal_cells"].values

# Compute latent time with manually set terminal cells
scv.tl.latent_time(adata2_subset, end_key="terminal_cells")


In [ ]:
scv.pl.scatter(adata1_subset, color="latent_time", cmap="coolwarm")


In [ ]:
scv.pl.scatter(adata2_subset, color="latent_time", cmap="coolwarm")


In [ ]:
# CellRANK velocitykernel

vk1 = cr.kernels.VelocityKernel(adata1_subset)


In [ ]:
vk2 = cr.kernels.VelocityKernel(adata2_subset)

In [ ]:
vk1.compute_transition_matrix()

In [ ]:
vk2.compute_transition_matrix()

In [ ]:
# combining gene expression similarity to reduce noise from RNA velocity

ck1 = cr.kernels.ConnectivityKernel(adata1_subset)
ck1.compute_transition_matrix()

In [ ]:
ck2 = cr.kernels.ConnectivityKernel(adata2_subset)
ck2.compute_transition_matrix()

In [ ]:
combined_kernel1 = 0.8 * vk1 + 0.2 * ck1

In [ ]:
combined_kernel2 = 0.8 * vk2 + 0.2 * ck2

In [ ]:
vk1.plot_projection(color="cluster_ids", save='vehstream2.png')


In [ ]:
vk2.plot_projection(color="cluster_ids", save='pthstream2.png')

In [ ]:
# computing inital and terminal states
g1 = cr.estimators.GPCCA(vk1)
print(g1)


In [ ]:
g2 = cr.estimators.GPCCA(vk2)
print(g2)

In [ ]:
g1.compute_macrostates(n_states=3, cluster_key="cluster_ids")
g1.plot_macrostates(which="all", legend_loc="right", s=100, save='vehallstates.png')

In [ ]:
g2.compute_macrostates(n_states=3, cluster_key="cluster_ids")
g2.plot_macrostates(which="all", legend_loc="right", s=100, save='pthallstates.png')

In [ ]:
g1.predict_terminal_states()
g1.plot_macrostates(which="terminal", legend_loc="right", s=100)

In [ ]:
g1.predict_initial_states()
g1.plot_macrostates(which="initial", legend_loc="right", s=100)

In [ ]:
g2.predict_terminal_states()
g2.plot_macrostates(which="terminal", legend_loc="right", s=100)

In [ ]:
g2.predict_initial_states()
g2.plot_macrostates(which="initial", legend_loc="right", s=100)

In [ ]:
g1.set_terminal_states(states=["Pre-osteoblast","CAR1"])
g2.set_terminal_states(states=["Pre-osteoblast","CAR1"])

In [ ]:
g1.set_initial_states(states=["CAR3"])
g1.plot_macrostates(which="initial", legend_loc="right", s=100, save='vehinitialstates.png')

In [ ]:
g2.set_initial_states(states=["CAR3"])
g2.plot_macrostates(which="initial", legend_loc="right", s=100, save='pthinitialstates.png')

In [ ]:
g1.plot_macrostates(which="terminal", legend_loc="right", s=100, save='vehterminalstates.png')


In [ ]:
g2.plot_macrostates(which="terminal", legend_loc="right", s=100, save='pthterminalstates.png')

In [ ]:
g1.compute_fate_probabilities()
g1.plot_fate_probabilities(same_plot=False)

In [ ]:
g2.compute_fate_probabilities()
g2.plot_fate_probabilities(same_plot=False)

In [ ]:
ob_states = ["CAR1", "CAR2", "CAR3", "CAR4", "CAR5", "CAR6", "Pre-osteoblast"]

sc.pl.embedding(adata2_subset, basis="umap", color="cluster_ids", groups=ob_states, legend_loc="right")

In [ ]:
cr.pl.aggregate_fate_probabilities(
    adata1_subset,
    mode="violin",
    lineages=["Pre-osteoblast"],
    cluster_key="cluster_ids",
    clusters=ob_states,
    xrot=45
)
plt.ylim(0,1)
plt.savefig("vehfate_probabilities_violin_preob.png", dpi=300, bbox_inches='tight')  # Save the plot

In [ ]:
cr.pl.aggregate_fate_probabilities(
    adata2_subset,
    mode="violin",
    lineages=["Pre-osteoblast"],
    cluster_key="cluster_ids",
    clusters=ob_states,
    xrot=45
)
plt.ylim(0,1)
plt.savefig("pthfate_probabilities_violin_preob.png", dpi=300, bbox_inches='tight')  # Save the plot

In [ ]:
import numpy as np
import pandas as pd

# Pre-osteoblast lineage (name)
lineage_name = "Pre-osteoblast"

# Function to extract fate probabilities from lineages_fwd
def extract_fate_probs(adata, lineage_name):
    fate_probs = []
    for p in adata.obsm["lineages_fwd"]:
        # If p is a Lineage object, use p[lineage_name]
        if hasattr(p, "__getitem__"):
            fate_probs.append(float(p[lineage_name]))
        else:
            fate_probs.append(float(p))  # fallback
    return np.array(fate_probs)

# Vehicle
veh_fp = extract_fate_probs(adata1_subset, lineage_name)
veh_cluster = adata1_subset.obs["cluster_ids"].astype(str)
df_veh = pd.DataFrame({
    "cluster": veh_cluster,
    "fate_prob": veh_fp,
    "condition": "Vehicle"
})

# PTH
pth_fp = extract_fate_probs(adata2_subset, lineage_name)
pth_cluster = adata2_subset.obs["cluster_ids"].astype(str)
df_pth = pd.DataFrame({
    "cluster": pth_cluster,
    "fate_prob": pth_fp,
    "condition": "PTH"
})

# Combine
df_all = pd.concat([df_veh, df_pth], ignore_index=True)


In [ ]:

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ---------------------------
# PARAMETERS
# ---------------------------
lineage_name = "Pre-osteoblast"  # lineage of interest
desired_order = ["CAR1", "CAR2", "CAR3", "CAR4", "CAR5", "CAR6", "Pre-osteoblast"]
condition_colors = {"Vehicle": "#1f77b4", "PTH": "#d62728"}  # colors for the violins

# ---------------------------
# FUNCTION TO EXTRACT FATE PROBS
# ---------------------------
def extract_fate_probs(adata, lineage_name):
    """
    Extract fate probabilities for a given lineage from adata.obsm['lineages_fwd'].
    Handles both arrays and Lineage objects.
    """
    fate_probs = []
    for p in adata.obsm["lineages_fwd"]:
        # If p is a Lineage object, use p[lineage_name]
        if hasattr(p, "__getitem__"):
            fate_probs.append(float(p[lineage_name]))
        else:
            fate_probs.append(float(p))  # fallback for arrays
    return np.array(fate_probs)

# ---------------------------
# EXTRACT VEHICLE
# ---------------------------
veh_fp = extract_fate_probs(adata1_subset, lineage_name)
veh_cluster = adata1_subset.obs["cluster_ids"].astype(str)

df_veh = pd.DataFrame({
    "cluster": veh_cluster,
    "fate_prob": veh_fp,
    "condition": "Vehicle"
})

# ---------------------------
# EXTRACT PTH
# ---------------------------
pth_fp = extract_fate_probs(adata2_subset, lineage_name)
pth_cluster = adata2_subset.obs["cluster_ids"].astype(str)

df_pth = pd.DataFrame({
    "cluster": pth_cluster,
    "fate_prob": pth_fp,
    "condition": "PTH"
})

# ---------------------------
# COMBINE DATA
# ---------------------------
df_all = pd.concat([df_veh, df_pth], ignore_index=True)

# Reorder clusters
df_all["cluster"] = pd.Categorical(df_all["cluster"], categories=desired_order, ordered=True)

# ---------------------------
# PLOT SIDE-BY-SIDE VIOLIN
# ---------------------------
plt.figure(figsize=(10, 5.5))

# Violin plot
sns.violinplot(
    data=df_all,
    x="cluster",
    y="fate_prob",
    hue="condition",
    dodge=True,
    inner=None,
    bw=1.5,
    cut=0,
    width=0.8,
    palette=condition_colors,
    linewidth=2  # make violin edges bolder
)

# Overlay individual points (jittered)
sns.stripplot(
    data=df_all,
    x="cluster",
    y="fate_prob",
    hue="condition",
    dodge=True,
    jitter=True,
    alpha=0.7,
    size=3,  # slightly bigger points for visibility
    palette=condition_colors,
    linewidth=1,  # bold edges for points
    edgecolor='black',  # optional: keep edges visible
    legend=False
)

# Axes and labels
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.ylabel("Fate Probability to Pre-osteoblast", fontsize=14, fontweight='bold')
plt.xlabel("Cluster", fontsize=14, fontweight='bold')

# Make x-axis labels bold and rotated
plt.xticks(rotation=45, fontsize=12, fontweight='bold')

# Make y-axis ticks bold
plt.yticks(fontsize=12, fontweight='bold')

# Make spines thicker
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(2)

# Make ticks thicker
plt.tick_params(width=2, labelsize=12)

# Legend at bottom right
plt.legend(title="Condition", loc='lower right', fontsize=12, title_fontsize=12)

plt.tight_layout()
plt.savefig("veh_vs_pth_fate_preob_sidebyside_cellrank_style_bold.png", dpi=300)
plt.show()



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 7))

sns.violinplot(
    data=df_all,
    x="cluster",
    y="fate_prob",
    hue="condition",
    inner="quartile",
    cut=0
)

plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("veh_vs_pth_fate_preob_split_violin.png", dpi=300)
plt.show()


In [ ]:
# subset to just mature osteoblasts cells
bdata1 = adata1_subset[adata1_subset.obs["cluster_ids"] == "Mature osteoblast"].copy()

# create an annotation for terminal vs. not-terminal
bdata1.obs["maturation_state"] = np.where(
    bdata1.obs["term_states_fwd"] == "Mature osteoblast", "terminal", "not terminal"
)

# show distribution in violin plot
import scanpy as sc
import matplotlib.pyplot as plt

# Your existing code to plot the violin plot
sc.pl.violin(
    bdata1, 
    keys=["Ebf1", "Ebf2", "Ebf3", "Runx1", "Runx2", "Foxc1"], 
    groupby="maturation_state", 
    show=False  # Don't immediately display, so we can adjust the axis
)

# Manually adjust the y-axis limits
plt.ylim(0, 2)  # Set y-axis range from 0 to 1.75

# Show the plot
plt.show()

import scipy.stats as st  # Import scipy.stats with alias 'st'
# use a simple t-test to quantify how different the two distributions are
a = bdata1[bdata1.obs["maturation_state"] == "terminal", "Foxc1"].X.data
b = bdata1[bdata1.obs["maturation_state"] == "not terminal", "Foxc1"].X.data

# Calculate and print the mean of both distributions
mean_a = np.mean(a)
mean_b = np.mean(b)

print(f"Mean of terminal cells: {mean_a}")
print(f"Mean of not terminal cells: {mean_b}")

# Perform the t-test
t_stat, p_value = st.ttest_ind(a, b, equal_var=False)

print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")


In [ ]:
# subset to just mature osteoblasts cells
bdata2 = adata2_subset[adata2_subset.obs["cluster_ids"] == "Mature osteoblast"].copy()

# create an annotation for terminal vs. not-terminal
bdata2.obs["maturation_state"] = np.where(
    bdata2.obs["term_states_fwd"] == "Mature osteoblast", "terminal", "not terminal"
)

# show distribution in violin plot
sc.pl.violin(
    bdata2, 
    keys=["Ebf1", "Ebf2", "Ebf3", "Runx1", "Runx2", "Foxc1"], 
    groupby="maturation_state", 
    show=False  # Don't immediately display, so we can adjust the axis
)

# Manually adjust the y-axis limits
plt.ylim(0, 2)  # Set y-axis range from 0 to 1.75

# Show the plot
plt.show()

import scipy.stats as st  # Import scipy.stats with alias 'st'
# use a simple t-test to quantify how different the two distributions are
a = bdata2[bdata2.obs["maturation_state"] == "terminal", "Foxc1"].X.data
b = bdata2[bdata2.obs["maturation_state"] == "not terminal", "Foxc1"].X.data

# Calculate and print the mean of both distributions
mean_a = np.mean(a)
mean_b = np.mean(b)

print(f"Mean of terminal cells: {mean_a}")
print(f"Mean of not terminal cells: {mean_b}")

# Perform the t-test
t_stat, p_value = st.ttest_ind(a, b, equal_var=False)

print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")



In [ ]:
cr.pl.aggregate_fate_probabilities(
    adata1_subset,
    mode="violin",
    lineages=["Mature osteoblast"],
    cluster_key="cluster_ids",
    clusters=ob_states,
    xrot=45
)
plt.ylim(0,1)

In [ ]:
driver_clusters = ["OsteoCAR1", "OsteoCAR2", "OsteoCAR3", "OsteoCAR4", "AdipoCAR", "Pre-osteoblast", "Mature osteoblast"]

ob_df1 = g1.compute_lineage_drivers(lineages=["Pre-osteoblast"], cluster_key="cluster_ids", clusters=driver_clusters)
ob_df1.head(50)



In [ ]:
driver_clusters = ["OsteoCAR1", "OsteoCAR2", "OsteoCAR3", "OsteoCAR4", "AdipoCAR", "Pre-osteoblast", "Mature osteoblast"]

ob_df2 = g2.compute_lineage_drivers(lineages=["Pre-osteoblast"], cluster_key="cluster_ids", clusters=driver_clusters)
ob_df2.head(50)

In [ ]:
adata1_subset.obs["fate_probabilities_osteoblast"] = g1.fate_probabilities["Pre-osteoblast"].X.flatten()

sc.pl.embedding(
    adata1_subset,
    basis="umap",
    color=["fate_probabilities_osteoblast"] + ["Malat1", "Ebf1", "Ebf3", "Runx1", "Runx2", "Adcy2"],
    color_map="viridis",
    s=10,
    ncols=3,
    vmax="p96",
)

In [ ]:
adata2_subset.obs["fate_probabilities_osteoblast"] = g2.fate_probabilities["Pre-osteoblast"].X.flatten()

sc.pl.embedding(
    adata2_subset,
    basis="umap",
    color=["fate_probabilities_osteoblast"] + ["Malat1", "Ebf1", "Ebf3", "Runx1", "Runx2", "Adcy2"],
    color_map="viridis",
    s=10,
    ncols=3,
    vmax="p96",
)

In [ ]:
adata1_subset.obs["fate_probabilities_osteoblast"] = g1.fate_probabilities["Mature osteoblast"].X.flatten()

sc.pl.violin(
    adata1_subset, 
    keys=["fate_probabilities_osteoblast", "Malat1"], 
    groupby="cluster_ids",  # Replace with a relevant categorical variable (e.g., clusters, cell types)
    jitter=True, 
    multi_panel=True,
    rotation=45
)


In [ ]:
adata2_subset.obs["fate_probabilities_osteoblast"] = g2.fate_probabilities["Mature osteoblast"].X.flatten()

sc.pl.violin(
    adata2_subset, 
    keys=["fate_probabilities_osteoblast", "Malat1"], 
    groupby="cluster_ids",  # Replace with a relevant categorical variable (e.g., clusters, cell types)
    jitter=True, 
    multi_panel=True,
    rotation=45
)


In [ ]:
ob_df1 = g1.compute_lineage_drivers()


# define set of genes to annotate
ob_genes = ["Malat1", "Ebf1", "Ebf3", "Runx1", "Runx2", "Adcy2"]
adipo_genes = ["Cxcl12", "Igfbp4", "Adipoq","Serping1"]

genes_oi = {
    "Pre-osteoblast": ob_genes,
    "AdipoCAR": adipo_genes,
}

# make sure all of these exist in AnnData
assert [
    gene in adata1_subset.var_names for genes in genes_oi.values() for gene in genes
], "Did not find all genes"

# compute mean gene expression across all cells
adata1_subset.var["mean expression"] = adata1_subset.X.mean(axis=0)

# visualize in a scatter plot
import matplotlib.pyplot as plt
import adjustText as at

# Generate the plot (without automatic text adjustment)
fig, ax = plt.subplots(figsize=(5, 5), dpi=150)
g1.plot_lineage_drivers_correlation(
    lineage_x="Pre-osteoblast",
    lineage_y="AdipoCAR",
    adjust_text=False,  # Disable automatic text adjustment
    gene_sets=genes_oi,
    color="mean expression",
    ax=ax,  # Use this ax object for manual control
    fontsize=10,
    size=50,
)

# Set x and y axis limits
ax.set_xlim(-0.4, 0.9)  # Set x-axis range to 0-1
ax.set_ylim(-0.8, 0.4)  # Set y-axis range to 0-1

# Get the colorbar and adjust its position
colorbar = ax.collections[0].colorbar  # Get the colorbar object
colorbar.ax.tick_params(labelsize=8)  # Adjust tick label size

# Move the colorbar to the right
# The pad controls the distance between the colorbar and the plot
# Use a larger fraction to adjust the size of the colorbar
fig.subplots_adjust(right=0.844)  # Adjust the right space of the entire figure
colorbar.set_label("Mean Expression", fontsize=10)
  # Adjust the position: [left, bottom, width, height]

# Retrieve text annotations from the plot
texts = ax.texts  # Get all text annotations from the axes

# Adjust text positions manually using adjustText
at.adjust_text(
    texts,  # The text objects you want to adjust
    ax=ax,  # Ensure the adjustments happen on the correct axes
    force_text=0.5,  # Force text to stay within plot bounds
    expand_text=(2, 2),  # Expand boundaries if necessary
    avoid_self=True,  # Avoid overlapping with itself
    prevent_crossings=True,  # Prevent text from crossing over each other
)

# Move the text farther from the point by adding offsets
for text in texts:
    x, y = text.get_position()  # Get the original position of the text
    text.set_position((x, y + 0.01))  # Move text farther from the point (both x and y)

# Display the plot
plt.show()

In [ ]:
ob_df2 = g2.compute_lineage_drivers()

# define set of genes to annotate
ob_genes = ["Malat1", "Ebf1", "Ebf3", "Runx1", "Runx2", "Adcy2"]
adipo_genes = ["Cxcl12", "Igfbp4", "Adipoq","Serping1"]

genes_oi = {
    "Pre-osteoblast": ob_genes,
    "AdipoCAR": adipo_genes,
}

# make sure all of these exist in AnnData
assert [
    gene in adata1_subset.var_names for genes in genes_oi.values() for gene in genes
], "Did not find all genes"

# compute mean gene expression across all cells
adata2_subset.var["mean expression"] = adata2_subset.X.mean(axis=0)


# visualize in a scatter plot
import matplotlib.pyplot as plt
import adjustText as at

# Generate the plot (without automatic text adjustment)
fig, ax = plt.subplots(figsize=(5, 5), dpi=150)
g2.plot_lineage_drivers_correlation(
    lineage_x="Pre-osteoblast",
    lineage_y="AdipoCAR",
    adjust_text=False,  # Disable automatic text adjustment
    gene_sets=genes_oi,
    color="mean expression",
    ax=ax,  # Use this ax object for manual control
    fontsize=10,
    size=50,
)

# Set x and y axis limits
ax.set_xlim(-0.4, 0.9)  # Set x-axis range to 0-1
ax.set_ylim(-0.8, 0.4)  # Set y-axis range to 0-1

# Get the colorbar and adjust its position
colorbar = ax.collections[0].colorbar  # Get the colorbar object
colorbar.ax.tick_params(labelsize=8)  # Adjust tick label size

# Move the colorbar to the right
# The pad controls the distance between the colorbar and the plot
# Use a larger fraction to adjust the size of the colorbar
fig.subplots_adjust(right=0.844)  # Adjust the right space of the entire figure
colorbar.set_label("Mean Expression", fontsize=10)
  # Adjust the position: [left, bottom, width, height]

# Retrieve text annotations from the plot
texts = ax.texts  # Get all text annotations from the axes

# Adjust text positions manually using adjustText
at.adjust_text(
    texts,  # The text objects you want to adjust
    ax=ax,  # Ensure the adjustments happen on the correct axes
    force_text=0.5,  # Force text to stay within plot bounds
    expand_text=(2, 2),  # Expand boundaries if necessary
    avoid_self=True,  # Avoid overlapping with itself
    prevent_crossings=True,  # Prevent text from crossing over each other
)

# Move the text farther from the point by adding offsets
for text in texts:
    x, y = text.get_position()  # Get the original position of the text
    text.set_position((x, y + 0.01))  # Move text farther from the point (both x and y)

# Display the plot
plt.show()


In [ ]:
sc.pl.umap(adata_subset, color = 'cluster_ids')

In [ ]:
umap = adata_subset.obsm['X_umap']
umap

In [ ]:
import sys, velocyto, scanpy, cellrank

print("Python:", sys.version)
print("velocyto:", velocyto.__version__)
print("scanpy:", scanpy.__version__)
print("cellrank:", cellrank.__version__)